# Regressão Linear com scikit-learn
## Ridge, Lasso e métricas de regressão

> Professor Carlos Eduardo Leal de Castro
> Disciplina: Métodos e Aplicações de IA


## 1. Preparação

Usaremos o dataset **Diabetes**, que já vem com o scikit-learn.

Ele contém medidas de pacientes e uma variável-alvo numérica relacionada à progressão da doença.

Não é necessário baixar nenhum arquivo externo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

RANDOM_STATE = 42

In [ ]:
dados = load_diabetes(as_frame=True)

X = dados.data
y = dados.target

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

X.head()

## 2. O que a regressão linear tenta aprender?

Para uma única variável, podemos pensar no modelo como uma reta:

$$
\hat{y} = \beta_0 + \beta_1 x
$$

Com várias variáveis:

$$
\hat{y} = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p
$$

- $\hat{y}$: valor previsto;
- $\beta_0$: intercepto;
- $\beta_j$: coeficiente associado à variável \(x_j\).

O treinamento procura coeficientes que façam as previsões ficarem o mais próximas possível dos valores reais.

Na regressão linear clássica, isso é feito minimizando a **soma dos erros quadráticos**:

$$
\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

O termo \(y_i-\hat{y}_i\) é chamado de **resíduo**.

### Separação treino/teste

Não devemos avaliar o modelo nos mesmos dados usados para treiná-lo.

Vamos usar:

- **80%** dos dados para treino;
- **20%** para teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Treino:", X_train.shape)
print("Teste :", X_test.shape)

## 3. Primeiro modelo com `LinearRegression`

No scikit-learn, o fluxo básico é quase sempre:

1. criar o modelo;
2. chamar `.fit(X_train, y_train)`;
3. chamar `.predict(X_test)`;
4. avaliar as previsões.

In [ ]:
modelo_linear = LinearRegression()

modelo_linear.fit(X_train, y_train)

y_pred_linear = modelo_linear.predict(X_test)

print("Intercepto:", modelo_linear.intercept_)

In [ ]:
coeficientes = pd.Series(
    modelo_linear.coef_,
    index=X.columns,
    name="coeficiente"
)

coeficientes.sort_values()

### Como interpretar um coeficiente?

Mantendo as outras variáveis constantes:

- coeficiente **positivo**: valores maiores da variável tendem a aumentar a previsão;
- coeficiente **negativo**: valores maiores tendem a diminuir a previsão;
- coeficiente próximo de **zero**: pouca contribuição linear, dadas as outras variáveis.

**Cuidado:** coeficiente não implica causalidade.

## 4. Métricas de regressão

Vamos usar quatro métricas.

### MAE — Mean Absolute Error

$$
MAE = \frac{1}{n}\sum |y_i-\hat{y}_i|
$$

É o erro absoluto médio. Quanto menor, melhor.

### MSE — Mean Squared Error

$$
MSE = \frac{1}{n}\sum (y_i-\hat{y}_i)^2
$$

Penaliza mais fortemente erros grandes. Quanto menor, melhor.

### RMSE — Root Mean Squared Error

$$
RMSE = \sqrt{MSE}
$$

Tem a mesma unidade da variável-alvo. Quanto menor, melhor.

### R² — Coeficiente de determinação

Mede quanto da variabilidade dos dados é explicada pelo modelo.

- $R^2 = 1$: ajuste perfeito;
- $R^2 = 0$: aproximadamente tão bom quanto prever sempre a média;
- $R^2 < 0$: pior do que esse baseline.

Quanto maior, melhor.

In [ ]:
def metricas_regressao(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R²": r2_score(y_true, y_pred)
    }

pd.Series(metricas_regressao(y_test, y_pred_linear)).round(3)

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_linear, alpha=0.7)

limites = [
    min(y_test.min(), y_pred_linear.min()),
    max(y_test.max(), y_pred_linear.max())
]
plt.plot(limites, limites, linestyle="--")

plt.xlabel("Valor real")
plt.ylabel("Valor previsto")
plt.title("Regressão Linear: real vs. previsto")
plt.show()

## 5. Por que regularizar?

Quando um modelo possui muitos coeficientes, alguns deles podem assumir valores muito grandes e se ajustar excessivamente aos dados de treino.

A **regularização** adiciona uma penalização ao processo de treinamento.

A intensidade dessa penalização é controlada por
$
\alpha
$

- `alpha = 0`: sem regularização;
- `alpha` maior: penalização mais forte.

Duas técnicas muito comuns são **Ridge** e **Lasso**.

## 6. Ridge — regularização L2

O Ridge penaliza o **quadrado** dos coeficientes:

$$
\text{Erro} + \alpha \sum_j \beta_j^2
$$
Efeito típico:

- reduz a magnitude dos coeficientes;
- normalmente não os leva exatamente a zero;
- pode ajudar quando há variáveis correlacionadas;
- reduz variância ao custo de introduzir algum viés.

Como a penalização depende da magnitude dos coeficientes, é boa prática **padronizar as variáveis** antes da regularização.

Por isso usaremos um `Pipeline`.

In [ ]:
modelo_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

modelo_ridge.fit(X_train, y_train)
y_pred_ridge = modelo_ridge.predict(X_test)

pd.Series(metricas_regressao(y_test, y_pred_ridge)).round(3)

## 7. Lasso — regularização L1

O Lasso penaliza o **valor absoluto** dos coeficientes:

$$
\text{Erro} + \alpha \sum_j |\beta_j|
$$

Uma diferença importante é que o Lasso pode levar alguns coeficientes exatamente a **zero**.

Isso faz com que ele também possa funcionar como uma forma simples de **seleção de variáveis**.

In [ ]:
modelo_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=1.0, max_iter=10000))
])

modelo_lasso.fit(X_train, y_train)
y_pred_lasso = modelo_lasso.predict(X_test)

pd.Series(metricas_regressao(y_test, y_pred_lasso)).round(3)

## 8. Comparando os três modelos

Nenhuma técnica é automaticamente melhor.

A comparação deve ser feita sobre dados não utilizados durante o treinamento.

In [ ]:
resultados = pd.DataFrame({
    "LinearRegression": metricas_regressao(y_test, y_pred_linear),
    "Ridge": metricas_regressao(y_test, y_pred_ridge),
    "Lasso": metricas_regressao(y_test, y_pred_lasso)
}).T

resultados.round(3)

### Coeficientes após regularização

Como Ridge e Lasso foram treinados com dados padronizados, seus coeficientes podem ser comparados entre si com mais segurança.

Observe principalmente o que acontece com o Lasso.

In [ ]:
coef_ridge = modelo_ridge.named_steps["model"].coef_
coef_lasso = modelo_lasso.named_steps["model"].coef_

comparacao_coef = pd.DataFrame({
    "Ridge": coef_ridge,
    "Lasso": coef_lasso
}, index=X.columns)

comparacao_coef.round(3)

In [ ]:
comparacao_coef.plot(kind="bar", figsize=(10, 4))
plt.axhline(0, linewidth=1)
plt.ylabel("Coeficiente")
plt.title("Coeficientes: Ridge vs. Lasso")
plt.tight_layout()
plt.show()

## 9. O papel do `alpha`

O `alpha` controla o equilíbrio entre:

- ajustar os dados;
- manter os coeficientes menores.

Valores muito pequenos produzem pouca regularização.

Valores muito grandes podem simplificar demais o modelo e causar **underfitting**.

Abaixo fazemos uma comparação simples usando alguns valores de `alpha`.

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

linhas = []

for alpha in alphas:
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])

    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)

    linhas.append({
        "alpha": alpha,
        **metricas_regressao(y_test, pred)
    })

resultado_alphas = pd.DataFrame(linhas)

resultado_alphas.round(3)

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogx(
    resultado_alphas["alpha"],
    resultado_alphas["RMSE"],
    marker="o"
)
plt.xlabel("alpha")
plt.ylabel("RMSE")
plt.title("Ridge: efeito do alpha")
plt.grid(True)
plt.show()

## 10. Resumo

### `LinearRegression`

Modelo linear sem penalização explícita.

### `Ridge`

- regularização **L2**;
- reduz os coeficientes;
- dificilmente zera coeficientes.

### `Lasso`

- regularização **L1**;
- pode zerar coeficientes;
- pode realizar seleção de variáveis.

### Métricas

| Métrica | Melhor direção | Interpretação resumida |
|---|---:|---|
| MAE | ↓ | erro absoluto médio |
| MSE | ↓ | erro quadrático médio |
| RMSE | ↓ | erro na unidade do alvo |
| R² | ↑ | proporção da variabilidade explicada |

> Em um problema real, o modelo deve ser escolhido considerando simultaneamente desempenho, estabilidade, interpretabilidade e validação adequada.

# Exercícios

Faça os exercícios abaixo alterando apenas as células marcadas com `TODO`.

Tente primeiro sem consultar as células anteriores em excesso. O objetivo é consolidar o fluxo completo.

## Exercício 1 — Treinar uma regressão linear

Crie uma nova divisão treino/teste usando `random_state=7`.

Depois:

1. treine `LinearRegression`;
2. gere previsões para o conjunto de teste;
3. calcule o RMSE.

**TODO:** complete a célula abaixo.

In [ ]:
# TODO 1

# X_train_ex1, X_test_ex1, y_train_ex1, y_test_ex1 = ...
# modelo_ex1 = ...
# ...

## Exercício 2 — Interpretar métricas

Usando o modelo do exercício anterior, calcule:

- MAE;
- MSE;
- RMSE;
- R².

Depois responda nos comentários:

1. qual é o erro médio aproximado em unidades do alvo?
2. qual métrica é mais diretamente útil para responder essa pergunta?

**TODO:** complete a célula abaixo.

In [ ]:
# TODO 2

# metricas_ex2 = ...
# print(metricas_ex2)

# Resposta:
# 1.
# 2.

## Exercício 3 — Comparar valores de `alpha` no Ridge

Treine modelos Ridge para:

```python
alphas = [0.01, 0.1, 1, 10, 100]
```

Use `StandardScaler` + `Ridge` em um `Pipeline`.

Encontre o `alpha` com menor RMSE no conjunto de teste.

**TODO:** complete a célula abaixo.

In [ ]:
# TODO 3

alphas_ex3 = [0.01, 0.1, 1, 10, 100]

# resultados_ex3 = []
# for alpha in alphas_ex3:
#     ...

## Exercício 4 — Lasso e seleção de variáveis

Treine um `Lasso` com `alpha=5`.

Depois descubra:

1. quantos coeficientes ficaram exatamente iguais a zero;
2. quais variáveis foram zeradas.

Use um `Pipeline` com `StandardScaler`.

**TODO:** complete a célula abaixo.

In [ ]:
# TODO 4

# modelo_ex4 = ...
# modelo_ex4.fit(...)
# coef_ex4 = ...
# ...

## Exercício 5 — Escolha final do modelo

Compare, usando a **mesma divisão treino/teste**:

- `LinearRegression`;
- o melhor Ridge encontrado no Exercício 3;
- `Lasso(alpha=1)`.

Monte um `DataFrame` com **MAE, RMSE e R²**.

Depois escreva, em 2–4 linhas nos comentários, qual modelo você escolheria e por quê.

**TODO:** complete a célula abaixo.

In [ ]:
# TODO 5

# comparacao_final = ...
# display(comparacao_final)

# Minha escolha:
#
#